In [0]:
%sql
USE dbjahan.silver_data;

In [0]:
%sql
SELECT * FROM dbjahan.Silver_data.elpris


elomrade,ar,snitt_sek_per_kwh,n_timmar
SE2,2025,0.21845232079027763,15387
SE3,2025,0.5557320647299667,15387
SE4,2025,0.6921790037044256,15387
SE1,2025,0.22366247936569844,15387


In [0]:
%sql
SELECT * FROM dbjahan.Silver_data.nyreg_bilar
LIMIT 10;

kommun_kod,kommun_namn,drivmedel_kod,drivmedel_namn,ar,manad,tid,antal
0114,Upplands Väsby,100,bensin,2022,1,2022M01,17
0114,Upplands Väsby,100,bensin,2022,2,2022M02,13
0114,Upplands Väsby,100,bensin,2022,3,2022M03,18
0114,Upplands Väsby,100,bensin,2022,4,2022M04,18
0114,Upplands Väsby,100,bensin,2022,5,2022M05,25
0114,Upplands Väsby,100,bensin,2022,6,2022M06,17
0114,Upplands Väsby,100,bensin,2022,7,2022M07,13
0114,Upplands Väsby,100,bensin,2022,8,2022M08,17
0114,Upplands Väsby,100,bensin,2022,9,2022M09,18
0114,Upplands Väsby,100,bensin,2022,10,2022M10,9


In [0]:
%sql
SELECT DISTINCT region, region_text
FROM dbjahan.bronze_data.nyreg_bilar_raw
WHERE length(region) = 4
ORDER BY region;

region,region_text
0114,Upplands Väsby
0115,Vallentuna
0117,Österåker
0120,Värmdö
0123,Järfälla
0125,Ekerö
0126,Huddinge
0127,Botkyrka
0128,Salem
0136,Haninge


In [0]:
%sql
CREATE OR REPLACE TABLE dbjahan.silver_data.kommun_elomrade AS
SELECT DISTINCT
  region AS kommun_kod,
  region_text AS kommun_namn,
  CASE
    WHEN region IN ('2180', '2104', '2181') THEN 'SE3'  -- Gävle, Hofors, Sandviken
    WHEN region IN ('2418', '2417', '2482') THEN 'SE1'  -- Malå, Norsjö, Skellefteå
    WHEN region IN ('1383', '1384') THEN 'SE3'          -- Varberg, Kungsbacka
    WHEN region IN ('0617', '0683') THEN 'SE4'          -- Gnosjö, Värnamo
    WHEN region IN ('0860', '0883', '0884') THEN 'SE3'  -- Hultsfred, Västervik, Vimmerby
    WHEN left(region, 2) = '25' THEN 'SE1'
    WHEN left(region, 2) IN ('24', '23', '22', '21') THEN 'SE2'
    WHEN left(region, 2) IN ('07', '08', '10', '12', '13') THEN 'SE4'
    ELSE 'SE3'
  END AS elomrade
FROM dbjahan.bronze_data.nyreg_bilar_raw
WHERE length(region) = 4
  AND region <> '1917';

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT elomrade, COUNT(*) 
FROM dbjahan.Silver_data.kommun_elomrade
GROUP BY elomrade;

elomrade,COUNT(*)
SE3,185
SE4,52
SE2,40
SE1,14


In [0]:
%sql
SELECT k.kommun_namn, k.elomrade, p.snitt_sek_per_kwh
FROM dbjahan.Silver_data.kommun_elomrade k
LEFT JOIN dbjahan.Silver_data.elpris p
  ON k.elomrade = p.elomrade
WHERE k.kommun_kod IN ('0180', '1280', '2584');

kommun_namn,elomrade,snitt_sek_per_kwh
Stockholm,SE3,0.5557320647299667
Malmö,SE4,0.6921790037044256
Kiruna,SE1,0.22366247936569844
